# CSI Groupcast Overview Analysis

Analyses the Parquet output produced by `run_csi_sweep.py` for the four schemes:
- **S1** Fixed-MCS baseline
- **S2** Full-CSI upper bound
- **S3** Naive partial-CSI (random K reporters, min-CQI adapter)
- **S4** Proposed (edge-biased reporters, ExpectedGPRR adapter)

## Data caveat

> The CSI feedback is modelled as an **idealized in-memory registry** (no PSFCH/CSI-RS PDUs
> on the air). S2 (all reporters) is a clean upper bound; S3/S4 performance reflects only
> the *policy* quality, not feedback-channel noise or overhead. Real-world performance will
> be slightly worse due to PSFCH reliability and timing.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.1)
SCHEME_ORDER = ['S1', 'S2', 'S3', 'S4']
SCHEME_PALETTE = dict(zip(SCHEME_ORDER, sns.color_palette('tab10', 4)))

# ---- Load data ---------------------------------------------------------------
SWEEP_DIR = Path('../../../../tmp/csi_sweep')
parquet_run = SWEEP_DIR / '_kpi_summary.parquet'
parquet_rx  = SWEEP_DIR / '_kpi_summary_per_rx.parquet'

assert parquet_run.exists(), f'Run Parquet not found: {parquet_run}'
assert parquet_rx.exists(),  f'Per-rx Parquet not found: {parquet_rx}'

df_run = pd.read_parquet(parquet_run)
df_rx  = pd.read_parquet(parquet_rx)

# Numeric coercions.
for col in ['prr', 'pir_mean_s', 'pir_p90_s', 'pir_p95_s', 'throughput_kbps',
            'group_prr', 'gprr', 'mcs_mean', 'fb_reporters_mean',
            'distance_m', 'n_ues', 'feedback_k', 'ivd_m', 'mcr_m', 'speed_mps']:
    for df in [df_run, df_rx]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

df_run['scheme'] = pd.Categorical(df_run['scheme'], categories=SCHEME_ORDER, ordered=True)
df_rx ['scheme'] = pd.Categorical(df_rx ['scheme'], categories=SCHEME_ORDER, ordered=True)

print(f'df_run: {df_run.shape}   schemes: {sorted(df_run.scheme.unique())}')
print(f'df_rx:  {df_rx.shape}    scenarios: {sorted(df_run.scenario.unique())}')
print(df_run.groupby('scheme')['gprr'].describe().round(3))

## §1 — Per-receiver PRR vs Distance

Each point is one (rx_node, run). A receiver at distance **d** from the group Tx receives
a fraction PRR of the transmitted packets. Lower MCS (S1-fixed) may over-protect or
under-protect; adaptive schemes (S2/S4) should push the curve toward PRR=1 at the MCR.

In [ ]:
df_rx_ok = df_rx.dropna(subset=['prr', 'distance_m'])
df_rx_ok = df_rx_ok[df_rx_ok['distance_m'] >= 0]

for scen in df_rx_ok['scenario'].unique():
    sub = df_rx_ok[df_rx_ok['scenario'] == scen]
    g = sns.relplot(
        data=sub, x='distance_m', y='prr',
        hue='scheme', hue_order=SCHEME_ORDER,
        col='n_ues', row='mu',
        kind='line', estimator='median', errorbar=('pi', 90),
        height=3, aspect=1.2,
        palette=SCHEME_PALETTE,
    )
    g.set_axis_labels('Distance from Tx (m)', 'PRR')
    g.figure.suptitle(f'Scenario: {scen} — Per-Rx PRR vs Distance (median ± 90th-pctile)', y=1.02)
    plt.tight_layout()
    plt.show()
    print(f'[{scen}] PRR at MCR (last decile by distance):',
          sub.groupby('scheme').apply(
              lambda d: d[d['distance_m'] >= d['distance_m'].quantile(0.8)]['prr'].median()
          ).round(3).to_dict())

## §2 — GPRR vs Group Size

**GPRR** (Group Packet Reception Ratio) = fraction of packets received by *all* members.
As group size grows, every additional receiver is another failure point, so GPRR drops.
An adaptive scheme should keep GPRR high by choosing an MCS that balances rate vs. coverage.

In [ ]:
g = sns.relplot(
    data=df_run.dropna(subset=['gprr']),
    x='n_ues', y='gprr',
    hue='scheme', hue_order=SCHEME_ORDER,
    col='scenario', row='mu',
    kind='line', estimator='median', errorbar=('pi', 90),
    height=3.2, aspect=1.3,
    palette=SCHEME_PALETTE,
    markers=True, dashes=False,
)
g.set_axis_labels('Group Size N (UEs)', 'GPRR')
g.figure.suptitle('GPRR vs Group Size (median ± 90th-pctile, across seeds × IVD × MCS)', y=1.02)
plt.tight_layout()
plt.show()

## §3 — PIR CDF by Scheme

The 95th-percentile PIR is the worst-case information staleness: 95% of inter-reception
gaps are below this value. The floor is the flow RRI (20 ms for platoon control).
Values > 2×RRI imply ≥1 missed packet; > 5×RRI implies consecutive losses.

In [ ]:
fig, axes = plt.subplots(1, len(df_rx['scenario'].unique()), figsize=(5 * len(df_rx['scenario'].unique()), 4))
if not hasattr(axes, '__iter__'):
    axes = [axes]

for ax, scen in zip(axes, sorted(df_rx['scenario'].unique())):
    sub = df_rx[(df_rx['scenario'] == scen) & df_rx['pir_p95_s'].notna()]
    for scheme, grp in sub.groupby('scheme'):
        vals = np.sort(grp['pir_p95_s'].dropna().values)
        cdf = np.arange(1, len(vals)+1) / len(vals)
        ax.plot(vals * 1000, cdf, label=str(scheme), color=SCHEME_PALETTE.get(str(scheme)))
    ax.axvline(20, color='grey', lw=1, ls='--', label='RRI floor (20 ms)')
    ax.set_xlabel('PIR P95 (ms)')
    ax.set_ylabel('CDF')
    ax.set_title(f'Scenario: {scen}')
    ax.set_xlim(0, 300)
    ax.legend()
plt.suptitle('CDF of P95 PIR by Scheme', y=1.02)
plt.tight_layout()
plt.show()
print('Median P95 PIR (ms):')
print(df_rx.groupby('scheme')['pir_p95_s'].median().mul(1000).round(2).to_dict())

## §4 — K-Budget vs GPRR

Shows how S3 (random reporters) and S4 (proposed) approach the S2 upper bound as K increases.
The gap at small K quantifies the value of the proposed reporter-selection policy.

In [ ]:
if 'feedback_k' in df_run.columns:
    sub = df_run[df_run['scheme'].isin(['S2', 'S3', 'S4'])].dropna(subset=['gprr', 'feedback_k'])
    g = sns.relplot(
        data=sub,
        x='feedback_k', y='gprr',
        hue='scheme', hue_order=['S2', 'S3', 'S4'],
        col='n_ues', row='scenario',
        kind='line', estimator='median', errorbar=('pi', 90),
        height=3, aspect=1.2,
        palette={s: SCHEME_PALETTE[s] for s in ['S2', 'S3', 'S4']},
        markers=True, dashes=False,
    )
    g.set_axis_labels('Feedback Budget K (reporters/tick)', 'GPRR')
    g.figure.suptitle('GPRR vs Feedback Budget K  (S2=full-CSI upper bound)', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('feedback_k column not in df_run — did the sweep include multiple K values?')

## §5 — Edge-User PRR

Edge users (receivers in the outer 20% of the MCR band) are the hardest to serve.
Adaptive schemes targeting the MCR boundary (EdgeBiased reporter) should show a
higher median PRR for edge users than S1 and S3.

In [ ]:
if 'is_edge' in df_rx.columns:
    edge_rx = df_rx[(df_rx['is_edge'].astype(float) == 1) & df_rx['prr'].notna()]
    fig, axes = plt.subplots(1, max(1, len(edge_rx['scenario'].unique())),
                             figsize=(5 * max(1, len(edge_rx['scenario'].unique())), 4),
                             sharey=True)
    if not hasattr(axes, '__iter__'):
        axes = [axes]
    for ax, scen in zip(axes, sorted(edge_rx['scenario'].unique())):
        sns.boxplot(data=edge_rx[edge_rx['scenario'] == scen],
                    x='scheme', y='prr', order=SCHEME_ORDER,
                    palette=SCHEME_PALETTE, ax=ax, showfliers=False)
        ax.set_title(f'{scen}')
        ax.set_xlabel('Scheme')
        ax.set_ylabel('PRR (edge receivers)')
        ax.axhline(0.99, color='red', lw=1, ls='--', label='QoS target 0.99')
        ax.legend()
    plt.suptitle('Edge-User PRR by Scheme (within outer 20% of MCR)', y=1.02)
    plt.tight_layout()
    plt.show()
    print('Median edge PRR by scheme:')
    print(edge_rx.groupby('scheme')['prr'].median().round(3).to_dict())
else:
    print('is_edge column not available in df_rx')

## §6 — Resource Efficiency: Mean MCS Used by Scheme

Higher mean MCS = higher spectral efficiency = more payload per symbol.
S1 is fixed; S4 should use a higher MCS than S3 by exploiting better reporter selection
to accurately identify how high the MCS can safely go.

In [ ]:
if 'mcs_mean' in df_run.columns:
    mcs_df = df_run.dropna(subset=['mcs_mean'])
    g = sns.catplot(
        data=mcs_df, x='scheme', y='mcs_mean',
        order=SCHEME_ORDER,
        col='n_ues',
        kind='box', palette=SCHEME_PALETTE,
        showfliers=False, height=3.5, aspect=0.8,
    )
    g.set_axis_labels('Scheme', 'Mean MCS used')
    g.figure.suptitle('Mean MCS per Scheme (higher = more spectrally efficient)', y=1.02)
    plt.tight_layout()
    plt.show()
    print('Median mean-MCS by scheme:')
    print(mcs_df.groupby('scheme')['mcs_mean'].median().round(2).to_dict())
else:
    print('mcs_mean column not in df_run')

## §7 — HARQ ACK/NACK Totals by Scheme

NACK rate = NACK / (ACK + NACK). A well-calibrated adaptive scheme keeps the NACK rate
near the configured BLER target (10%). S4 should match or beat S3; both should be below S1
when S1's fixed MCS is aggressive.

In [ ]:
harq_df = df_run.dropna(subset=['harq_ack_total', 'harq_nack_total']).copy()
harq_df['harq_ack_total']  = pd.to_numeric(harq_df['harq_ack_total'],  errors='coerce')
harq_df['harq_nack_total'] = pd.to_numeric(harq_df['harq_nack_total'], errors='coerce')
harq_df = harq_df.dropna(subset=['harq_ack_total', 'harq_nack_total'])
denom = harq_df['harq_ack_total'] + harq_df['harq_nack_total']
harq_df['nack_rate'] = harq_df['harq_nack_total'] / denom.replace(0, float('nan'))

g = sns.catplot(
    data=harq_df.dropna(subset=['nack_rate']),
    x='scheme', y='nack_rate',
    order=SCHEME_ORDER,
    col='n_ues',
    kind='box', palette=SCHEME_PALETTE,
    showfliers=False, height=3.5, aspect=0.8,
)
for ax in g.axes.flat:
    ax.axhline(0.10, color='red', lw=1, ls='--', label='10% BLER target')
    ax.legend()
g.set_axis_labels('Scheme', 'HARQ NACK rate')
g.figure.suptitle('HARQ NACK Rate by Scheme (target ≈ 10%)', y=1.02)
plt.tight_layout()
plt.show()
print('Median NACK rate by scheme:')
print(harq_df.groupby('scheme')['nack_rate'].median().round(3).to_dict())

## §8 — Summary Table

Aggregate leaderboard: median GPRR, group PRR, P95 PIR, mean MCS, NACK rate — by scheme.

In [ ]:
agg_cols = {c: 'median' for c in ['group_prr', 'gprr', 'mcs_mean'] if c in df_run.columns}
if 'nack_rate' in harq_df.columns:
    df_run_h = df_run.merge(harq_df[['stem', 'nack_rate']], on='stem', how='left')
    agg_cols['nack_rate'] = 'median'
else:
    df_run_h = df_run

pir_agg = df_rx.groupby('scheme')['pir_p95_s'].median().mul(1000).rename('pir_p95_ms')

summary = df_run_h.groupby('scheme').agg(agg_cols).round(3)
summary = summary.join(pir_agg.round(1))
print('\n=== Scheme Leaderboard (medians across all conditions) ===')
print(summary.to_string())